# Secondary-school sectors

This notebook keeps mutually-exclusive parliamentarian categories separate from school-attendance relationship counts.

In [2]:
# ruff: noqa: E402
import os
import sys
from pathlib import Path

import pandas as pd

root = Path(os.environ.get("APEMAP_PROJECT_ROOT", Path.cwd())).resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

from apemap.analysis import compute_sector_summary
from apemap.db import get_connection

parliament = int(os.environ.get("APEMAP_PARLIAMENT", "48"))
db_path = Path(
    os.environ.get("APEMAP_DB_PATH", root / "data" / "aped.duckdb")
).resolve()
conn = get_connection(db_path, read_only=True)
sectors = compute_sector_summary(conn, parliament)

In [6]:
for parliament_int in range(46, 48 + 1):
    sectors = compute_sector_summary(conn, f"{parliament_int}")
    print(f"Parliament {parliament_int}")
    unique = pd.DataFrame(
        [
            {
                "sector": sector,
                "parliamentarians": count,
                "percentage_of_known": sectors[
                    "percentage_of_known_parliamentarians"
                ].get(sector),
            }
            for sector, count in sectors["unique_parliamentarians_by_sector"].items()
        ]
    )
    display(unique)

Parliament 46


,sector,parliamentarians,percentage_of_known
0,Government,54,27.272727
1,Catholic,29,14.646465
2,Independent,28,14.141414
3,Combined/Multiple,80,40.404040
4,Other,7,3.535354
5,No School Recorded,40,NaN


Parliament 47


,sector,parliamentarians,percentage_of_known
0,Government,52,26.0
1,Catholic,27,13.5
2,Independent,31,15.5
3,Combined/Multiple,81,40.5
4,Other,9,4.5
5,No School Recorded,37,NaN


Parliament 48


,sector,parliamentarians,percentage_of_known
0,Government,48,26.519337
1,Catholic,20,11.049724
2,Independent,29,16.022099
3,Combined/Multiple,75,41.436464
4,Other,9,4.972376
5,No School Recorded,49,NaN


In [7]:
attendance = pd.DataFrame(
    [
        {
            "sector": sector,
            "attendance_instances": count,
            "percentage_of_instances": sectors["percentage_of_attendance_instances"][
                sector
            ],
        }
        for sector, count in sectors["attendance_instances_by_sector"].items()
    ]
)
attendance

,sector,attendance_instances,percentage_of_instances
0,Government,116,35.802469
1,Catholic,67,20.679012
2,Independent,52,16.049383
3,Other,89,27.469136


In [8]:
conn.close()